# Lecture 7 — Final Matrix Completion

In the previous notebook, we compared candidate combinations of latent rank $k$ and regularization strength $\lambda$ and selected the best pair $(k^{*},\lambda^{*})$. The next step is to use those selected hyperparameters in a final matrix-factorization model.

This notebook demonstrates the workflow on the same synthetic setting:

```text
select (k, lambda)  →  fit the final model  →  reconstruct X  →  predict missing entries
```

The complete synthetic matrix is retained only so that we can visualize the predictions. In a real recommender system, genuinely missing ratings are unknown, so their true values cannot be used to calculate RMSE.

## 1. Generate the synthetic rating matrix

We generate a complete low-rank matrix $X_{\text{true}}$ and hide some of its entries. The factorization algorithm receives only the observed entries.

For this experiment, notebook 02 selected

$$
k^{*}=2,\qquad \lambda^{*}=0.0001.
$$

The hidden entries are used here only to illustrate what the model is trying to predict.

In [1]:
import numpy as np

from matrix_factorization import factorize, reconstruct

np.set_printoptions(precision=3, suppress=True)

In [2]:
rng = np.random.default_rng(42)
n_users, n_movies, k_true = 8, 7, 2
U_true = rng.normal(size=(n_users, k_true))
V_true = rng.normal(size=(n_movies, k_true))
X_true = U_true @ V_true.T

mask = rng.random((n_users, n_movies)) < 0.55
Y_synth = np.where(mask, X_true, np.nan)

print('Observed matrix Y:')
print(Y_synth)

Observed matrix Y:
[[   nan    nan  0.652  0.533  0.236 -0.218 -0.322]
 [-0.625  0.612    nan  0.772    nan    nan  0.715]
 [   nan    nan  1.247 -2.184    nan -1.514 -1.366]
 [ 0.35     nan    nan    nan    nan -0.048    nan]
 [ 0.812  0.028    nan  0.111    nan    nan    nan]
 [   nan    nan -0.692  0.955 -0.651  0.752  0.698]
 [   nan  0.002 -0.78  -0.093 -0.425    nan    nan]
 [   nan    nan  0.499    nan  0.102 -0.065 -0.177]]


## 2. Fit the final model with the selected hyperparameters

Now that $k^{*}$ and $\lambda^{*}$ have been selected, we no longer compare candidate models. We fit one final model using those values and **all ratings available for training**.

Conceptually, the final fitting step is:

$$
U^{*},V^{*}\leftarrow\text{factorize}(Y,k^{*},\lambda^{*}).
$$

In this synthetic example, `Y_synth` contains all the ratings designated as observed. In a real experiment, this corresponds to fitting the final model after model selection using the available training and validation ratings.

In [3]:
best_k = 2
best_lambda = 0.0001

U_final, V_final, history = factorize(
    Y_synth,
    k=best_k,
    lambda_=best_lambda,
    seed=1,
)

X_hat = reconstruct(U_final, V_final)

print(f'Selected k: {best_k}')
print(f'Selected lambda: {best_lambda:g}')
print('U_final:')
print(U_final)
print('\nV_final:')
print(V_final)
print('\nX_hat:')
print(X_hat)

Selected k: 2
Selected lambda: 0.0001
U_final:
[[ 1.642 -2.381]
 [ 1.837  2.844]
 [-5.498 -4.555]
 [ 0.714 -0.775]
 [ 0.506 -2.335]
 [ 2.358  2.527]
 [-0.495  2.845]
 [ 2.028 -1.813]]

V_final:
[[ 0.148 -0.315]
 [ 0.262  0.046]
 [ 0.    -0.274]
 [ 0.371  0.032]
 [-0.098 -0.166]
 [ 0.127  0.179]
 [ 0.087  0.195]]

X_hat:
[[ 0.994  0.322  0.653  0.533  0.235 -0.217 -0.322]
 [-0.625  0.612 -0.779  0.772 -0.653  0.743  0.715]
 [ 0.622 -1.652  1.247 -2.184  1.295 -1.515 -1.366]
 [ 0.35   0.152  0.213  0.24   0.059 -0.048 -0.089]
 [ 0.812  0.026  0.64   0.113  0.339 -0.353 -0.412]
 [-0.448  0.735 -0.692  0.955 -0.651  0.752  0.698]
 [-0.971  0.    -0.78  -0.093 -0.425  0.446  0.512]
 [ 0.872  0.449  0.497  0.694  0.103 -0.066 -0.178]]


## 3. Predict the missing entries

The reconstructed matrix $\hat X=U^{*}(V^{*})^{T}$ contains a prediction for every user-movie pair.

For an observed entry, we already have a real rating. For a missing entry, the corresponding value of $\hat X$ is the model's prediction.

We therefore keep observed ratings unchanged and fill only the missing positions from $\hat X$.

In [4]:
X_completed = Y_synth.copy()
X_completed[~mask] = X_hat[~mask]

print('Completed matrix:')
print(X_completed)

Completed matrix:
[[ 0.994  0.322  0.652  0.533  0.236 -0.218 -0.322]
 [-0.625  0.612 -0.779  0.772 -0.653  0.743  0.715]
 [ 0.622 -1.652  1.247 -2.184  1.295 -1.514 -1.366]
 [ 0.35   0.152  0.213  0.24   0.059 -0.048 -0.089]
 [ 0.812  0.028  0.64   0.111  0.339 -0.353 -0.412]
 [-0.448  0.735 -0.692  0.955 -0.651  0.752  0.698]
 [-0.971  0.002 -0.78  -0.093 -0.425  0.446  0.512]
 [ 0.872  0.449  0.499  0.694  0.102 -0.065 -0.177]]


## 4. Compare with the synthetic truth

Because this is a synthetic experiment, we can inspect the hidden entries alongside their true values. This is useful for understanding the experiment, but it is **not available for genuinely missing ratings in a real recommender system**.

The comparison below shows only the entries that were hidden from the model.

In [5]:
hidden_indices = np.argwhere(~mask)

print('Hidden entries:')
print('user  movie  true  predicted')
for a, i in hidden_indices:
    print(f'{a:4d}  {i:5d}  {X_true[a, i]:5.3f}  {X_hat[a, i]:9.3f}')

Hidden entries:
user  movie  true  predicted
   0      0  1.110      0.994
   0      1  0.320      0.322
   1      2  -0.779     -0.779
   1      4  -0.653     -0.653
   1      5  0.743      0.743
   2      0  0.529      0.622
   2      1  -1.649     -1.652
   2      4  1.294      1.295
   3      1  0.128      0.152
   3      2  0.192      0.213
   3      3  0.205      0.240
   3      4  0.057      0.059
   3      6  -0.083     -0.089
   4      2  0.584      0.640
   4      4  0.308      0.339
   4      5  -0.321     -0.353
   4      6  -0.374     -0.412
   5      0  -0.422     -0.448
   5      1  0.734      0.735
   6      0  -1.057     -0.971
   6      5  0.447      0.446
   6      6  0.513      0.512
   7      0  0.996      0.872
   7      1  0.454      0.449
   7      3  0.704      0.694


## 5. From the experiment to a real recommender system

The synthetic experiment makes the complete workflow visible:

1. Start with observed ratings.
2. Split known ratings into training, validation, and test sets when evaluating a real system.
3. Train candidate models for several combinations of $k$ and $\lambda$.
4. Select $(k^{*},\lambda^{*})$ using validation performance.
5. Refit the final model with the selected $k^{*}$ and $\lambda^{*}$ using all data available for final training.
6. Compute $\hat X=U^{*}(V^{*})^{T}$.
7. Use $\hat X_{ai}$ for user-movie pairs whose ratings are genuinely missing.
8. Use the untouched test set only for the final evaluation of the selected modeling procedure.

The key distinction is that validation/test ratings are temporarily hidden **known ratings**, while genuinely missing ratings have no known target value.

## What to remember

- $k$ is selected as a hyperparameter; it is not learned as an entry of $U$ or $V$.
- $\lambda$ is also a hyperparameter and controls the strength of regularization.
- After selecting $(k^{*},\lambda^{*})$, fit the final factorization with both selected values.
- $U^{*}$ and $V^{*}$ are the learned latent-factor matrices of the final model.
- $\hat X=U^{*}(V^{*})^{T}$ gives a prediction for every user-movie pair.
- Keep observed ratings and use predictions for genuinely missing entries.
- In the real world, the quality of genuinely missing predictions cannot be measured directly because their true ratings are unknown.